In [ ]:
# =============================================================
# OpenQSim Phase 0A — Post-sweep QAOA re-run
# =============================================================
# Run this AFTER the main sweep finishes (phase0a_resume.ipynb).
# It:
#   1. Deletes the 120 failed QAOA raw JSONs
#   2. Un-marks them in the checkpoint
#   3. Re-runs all 210 QAOA combos with the fixed generator
#   4. Re-assembles the final dataset and pushes to Kaggle
#
# Prerequisite: the main sweep must have run first and its raw/
# directory must still be at /kaggle/working/benchmark_outputs/raw/
# =============================================================

import os, sys, subprocess, shutil, json
from pathlib import Path

REPO_DIR     = '/kaggle/working/OpenQSim'
OUTPUT_DIR   = '/kaggle/working/benchmark_outputs'
CKPT_DIR     = '/kaggle/working/checkpoints'
DATASET_DIR  = f'{OUTPUT_DIR}/datasets/openqsim_v0.1-small'

# Reload Kaggle credentials (same secrets as the main sweep)
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
if not KAGGLE_USERNAME:
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'NVIDIA_API_KEY'):
            v = s.get_secret(k)
            if v: os.environ[k] = v
        KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
    except Exception as e:
        print(f'WARN secrets: {e}')
KAGGLE_DATASET = f'{KAGGLE_USERNAME}/openqsim-benchmarks'
print(f'[OK] Target: {KAGGLE_DATASET}')

# Verify repo is present (was cloned during main sweep)
if not os.path.exists(REPO_DIR):
    print('Repo not found — cloning...')
    subprocess.check_call(['git', 'clone',
        'https://github.com/lekkalaharsha/opensim-ai', REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])
    print('[OK] Repo updated (QAOA fix confirmed)')
sys.path.insert(0, REPO_DIR)

# Show current state before cleanup
raw_files = list(Path(f'{OUTPUT_DIR}/raw').glob('*.json'))
import json as _json
qaoa_failed = [f for f in raw_files
               if 'qaoa' in _json.loads(f.read_text()).get('circuit_name','').lower()
               and not _json.loads(f.read_text()).get('success')]
print(f'Raw records total:   {len(raw_files)}')
print(f'Failed QAOA records: {len(qaoa_failed)} (will be deleted and re-run)')

# Run the all-in-one re-run script
subprocess.check_call([
    sys.executable,
    f'{REPO_DIR}/scripts/rerun_qaoa.py',
    '--raw-dir',        f'{OUTPUT_DIR}/raw',
    '--ckpt-dir',       CKPT_DIR,
    '--dataset-dir',    DATASET_DIR,
    '--repo-dir',       REPO_DIR,
    '--kaggle-dataset', KAGGLE_DATASET,
])

print('\n=== Done. Check dataset at:', f'https://www.kaggle.com/datasets/{KAGGLE_DATASET} ===')
